In [ ]:
import os.path as osp
import random

import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.utils.data as data
import torchvision
from cv2.gapi import kernel
from sympy.abc import lamda

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [ ]:
seed = 1000
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# 이미지 정보
base_dir = '../../data/VOCdevkit/VOC2012/'
img_data_template = osp.join(base_dir, 'JPEGImages', '%s.jpg')
annopath_template = osp.join(base_dir, 'Annotations', '%s.xml')

In [ ]:
# 학습을 위한 이미지 지정 데이터
tr_ids = osp.join(base_dir, 'ImageSets/Main/train.txt')
val_ids = osp.join(base_dir, 'ImageSets/Main/val.txt')

In [ ]:
tr_img_list = []
tr_anno_list = []

for i in open(tr_ids):
    tr_img_list.append(img_data_template % i.strip())
    tr_anno_list.append(annopath_template % i.strip())

In [ ]:
tr_img_list

In [ ]:
val_img_list = []
val_anno_list = []

for i in open(val_ids):
    val_img_list.append(img_data_template % i.strip())
    val_anno_list.append(annopath_template % i.strip())

In [ ]:
val_img_list

In [ ]:
# 클래스 화 하기
def make_datapath_list(rootpath):
    img_data_template = osp.join(rootpath, 'JPEGImages', '%s.jpg')
    annopath_template = osp.join(rootpath, 'Annotations', '%s.xml')

    tr_ids = osp.join(rootpath, 'ImageSets/Main/train.txt')
    val_ids = osp.join(rootpath, 'ImageSets/Main/val.txt')

    tr_img_list = []
    tr_anno_list = []

    for i in open(tr_ids):
        tr_img_list.append(img_data_template % i.strip())
        tr_anno_list.append(annopath_template % i.strip())

    val_img_list = []
    val_anno_list = []

    for i in open(val_ids):
        val_img_list.append(img_data_template % i.strip())
        val_anno_list.append(annopath_template % i.strip())

    return tr_img_list, tr_anno_list, val_img_list, val_anno_list

In [ ]:
base_dir = '../../data/VOCdevkit/VOC2012/'
tr_img_list, tr_anno_list, val_img_list, val_anno_list = make_datapath_list(base_dir)

In [ ]:
ck_xml = tr_anno_list[0]

In [ ]:
ck_xml

In [ ]:
l = ['a', 'b']
l.index('b')

In [ ]:
xml = ET.parse(ck_xml).getroot()
width = 500
height = 442
class_name_list = {key:10}

for i in xml.iter('object'):
    difficult = int(i.find('difficult').text)
    if difficult == 1:
        continue
    bndbox = []
    name = i.find('name').text.lower().strip()
    bbox = i.find('bndbox')
    pts = ['xmin', 'ymin', 'xmax', 'ymax']
    for pt in pts:
        cur_pixel = int(bbox.find(pt).text)-1

        if pt == 'xmin' or pt == 'xmax':
            cur_pixel /= width
        else:
            cur_pixel /= height
        bndbox.append(cur_pixel)
    label_index = class_name_list.index(name)
    bndbox.append(label_index)

In [ ]:
class Anno_xml2list:
    def __init__(self,classes):
        self.classes=classes
    def __call__(self,xml_path,width,height):
        ret=[]
        xml=ET.parse(ck_xml).getroot()
        for i in xml.iter('object'):
            difficult=int(i.find('difficult').text)
            if difficult==1:
                continue
            bndbox=[]
            name=i.find('name').text.lower().strip()
            bbox=i.find('bndbox')
            pts=['xmin','ymin','xmax','ymax']
            for pt in pts:
                cur_pixel=int(bbox.find(pt).text)-1

                if pt=='xmin' or pt=='xmax':
                    cur_pixel /= width
                else:
                    cur_pixel /= height
                bndbox.append(cur_pixel)
            label_index=self.classes.index(name)
            bndbox.append(label_index)
            ret.append(bndbox)
        return np.array(ret)

In [ ]:
from pathlib import Path
f_data = Path(base_dir+'ImageSets/Main/')
set_v = set()
for i in f_data.iterdir():
    if '_' in i.name:
        set_v.add(i.name.split('_')[0])
sorted(list(set_v))

In [ ]:
voc_classes=['aeroplane', 'bicycle', 'bird', 'boat',
'bottle', 'bus', 'car', 'cat', 'chair',
'cow', 'diningtable', 'dog', 'horse',
'motorbike', 'person', 'pottedplant',
'sheep', 'sofa', 'train', 'tvmonitor']

transform_anno=Anno_xml2list(voc_classes)
idx=123
img_path=tr_img_list[idx]
img=cv2.imread(img_path)
h,w,c=img.shape
ann_img_tr=transform_anno(tr_anno_list[idx],w,h)
ann_img_tr

In [ ]:
class Compose:
    def __init__(self, transforms):
        self.transforms=transforms
    def __call__(self, img, boxes = None, label = None):
        for t in self.transforms:
            img, boxes, label = t(img, boxes, label)
        return img, boxes, label

class ConvertFromInts:
    def __call__(self, img, boxes = None, label = None):
        return img.astype(np.float32), boxes, label

class ToAbsoluteCoords:
    def __call__(self, img, boxes = None, label = None):
        h, w, c = img.shape
        boxes[:, 0] *= w
        boxes[:, 2] *= w
        boxes[:, 1] *= h
        boxes[:, 3] *= h
        return img, boxes, label

class ToPercentCoords:
    def __call__(self, img, boxes = None, label = None):
        h, w, c = img.shape
        boxes[:, 0] /= w
        boxes[:, 2] /= w
        boxes[:, 1] /= h
        boxes[:, 3] /= h
        return img, boxes, label

class Resize:
    def __init__(self, size = 300):
        self.size = size
    def __call__(self, img, boxes = None, label = None):
        cv2.resize(img, (self.size, self.size))
        return img, boxes, label

class ConvertColor:
    def __init__(self, c = 'BGR', tr = 'HSV'):
        self.transform = tr
        self.current = c
    def __call__(self, img, boxes = None, label = None):
        if self.current == 'BGR' and self.transform == 'HSV':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        elif self.current == 'HSV' and self.transform == 'BGR':
            img = cv2.cvtColor(img, cv2.COLOR_HSV2BGR)
        else:
            raise NotImplementedError
        return img, boxes, label

def interset(box_a, box_b):
    max_xy = np.minimum(box_a[:,2:], box_b[2:])
    min_xy = np.maximum(box_a[:,:2], box_b[2:2])
    inter = np.clip((max_xy - min_xy), 0, np.inf)

def jacquard_numpy(box_a, box_b):
    inter = (box_a, box_b)
    ares_a = (box_a[:,2]-box_a[:,0])*(box_a[:,3]-box_a[:,1])
    ares_b = (box_b[:,2]-box_b[:,0])*(box_b[:,3]-box_b[:,1])
    union = ares_a+ares_b-inter
    return inter/union

class RandomContrast:
    def __init__(self, lower = .5, upper = .5):
        self.lower = lower
        self.upper = upper
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            alpha = random.uniform(self.lower, self.upper)
            img += alpha
        return img, boxes, label

class RandomSaturation:
    def __init__(self, lower = .5, upper = .5):
        self.lower = lower
        self.upper = upper
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            img[:,:,1] *= random.uniform(self.lower, self.upper)
        return img, boxes, label

class RandomHue:
    def __init__(self, delta = 18.0):
        assert .0 <= delta >= 360.0
        self.delta = delta
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            img[:,:,0] += random.uniform(-self.delta, self.delta)
            img[:,:,][img[:,:,0] > 360.0] -= 360.0
            img[:,:,][img[:,:,0] < .0] += 360.0
        return img, boxes, label

class RandomBrightness:
    def __init__(self, delta = 18.0):
        assert .0 <= delta >= 255.0
        self.delta = delta
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            alpha = random.uniform(-self.delta, self.delta)
            img += alpha
        return img, boxes, label

class RandomLightNosie:
    def __init__(self):
        self.perms = ((0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0))
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            swap = self.perms[random.randint(len(self.perms))]
            img = img[:,:,swap]
        return img, boxes, label

class RandomSampleCrop:
    def __init__(self):
        self.sample_option = ((None, None),
                              (.1, None),
                              (.3, None),
                              (.7, None),
                              (.9,None), None)
    def __call__(self, img, boxes = None, label = None):
        h, w, c = img.shape
        while True:
            idx = np.random.randint(len(self.sample_option))
            mode = self.sample_option[idx]
            if mode is None:
                return img, boxes, label
            min_iou, max_iou = mode
            if min_iou is None:
                min_iou = float('-inf')
            if max_iou is None:
                max_iou = float('-inf')
            for _ in range(50):
                current_img = img
                tr_w = random.uniform(.3*w,w)
                tr_h = random.uniform(.3*h,h)
                if tr_h/tr_w<.5 or tr_h/tr_w>2:
                    continue
                left = random.uniform(w-tr_w)
                top = random.uniform(h-tr_h)

                rect = np.array([int(left), int(top), int(left+tr_w), int(top+tr_h)])
                overlap =jacquard_numpy(boxes, rect)
                if overlap.min()<min_iou and overlap.max()>max_iou:
                    continue
                current_img = current_img[rect[1]:rect[3], rect[0]:rect[2],:]
                centers = (boces[:,:2]+boxes[:,:2])/2.0

                m1 = (rect[0]<centers[:,0])*(rect[1]<centers[:,1])
                m2 = (rect[2]<centers[:,0])*(rect[3]<centers[:,1])
                mask = m1 * m2

                if not mask.any():
                    continue

                current_boxes = boxes[mask, :].copy()
                current_label = label[mask]

                current_boxes[:, :2] = np.maximum(current_boxes[:, :2], rect[:2])
                current_boxes[:, :2] -= rect[:2]
                current_boxes[:, 2:] = np.minimum(current_boxes[:, 2:], rect[2:])
                current_boxes[:, 2:] -= rect[2:]
                return current_img, current_boxes, current_label

class PhotometricDistort:
    def __init__(self):
        self.pd = [
            RandomContrast(),
            ConvertColor(),
            RandomSaturation(),
            RandomHue(),
            ConvertColor('HSV', 'BGR'),
            RandomContrast()
        ],
        self.rand_brightness = RandomBrightness()
        self.rand_lightness_noise = RandomLightNosie()

    def __call__(self, img, boxes = None, label = None):
        im = img.copy()
        im, boxes, label = self.rand_brightness(im, boxes, label)
        if random.randint(2):
            distort = Compose(self.pd[:-1])
        else:
            distort = Compose(self.pd[1:])
        im, boxes, label = distort(im, boxes, label)
        return self.rand_lightness_noise(im, boxes, label)

def main():
    pass

if __name__ == '__main__':
    main()

In [ ]:
from utils.data_a import *

In [ ]:
#데이터 로더 구조 정의
class Make_dataset_Transform:
    def __init__(self,input_size,color_mean):
        self.base_transform={
            'train':Compose([
                ConvertFromInts(), # int를 float으로 변환
                ToAbsoluteCoords(), # 어노테이션 데이터의 규격화
                PhotometricDistort(), # 랜덤한 색조 변경
                RandomSampleCrop(), # 이미지의 랜덤 샘플화(이미지 증강구조 추가 가능)
                ToPercentCoords(), # 데이터 패턴 규격화(0~1)
                Resize(input_size), # 이미지 크기 변경
                SubtractMeans(color_mean) # 평균값 차연산
            ]),
            'val':Compose([
                ConvertFromInts(), # int를 float으로 변환
                Resize(input_size), # 이미지 크기 변경
                SubtractMeans(color_mean) # 평균값 차연산
        ])
        }
    def __call__(self, img, phase, boxes, label):
        return self.base_transform[phase](img, boxes, label)

In [ ]:
idx = 0
img_path = tr_img_list[idx]
img = cv2.imread(img_path)
h, w, c = img.shape

transform_anno = Anno_xml2list(voc_classes)
anno_list = transform_anno(tr_anno_list[idx], w, h)

In [ ]:
color_mean = (104, 117, 123) # BGR 평균
input_size = 300

tr = Make_dataset_Transform(input_size, color_mean)

# 모델

In [ ]:
import os
import urllib.request

In [ ]:
weights_dir = '../../weights/'

if not os.path.exists(weights_dir):
    os.makedirs(weights_dir)

url = "https://s3.amazonaws.com/amdegroot-models/vgg16_reducedfc.pth"
t_path = os.path.join(weights_dir, 'vgg16_reducedfc.pth')
if not os.path.exists(t_path):
    urllib.request.urlretrieve(url, t_path)

In [ ]:
url = "https://s3.amazonaws.com/amdegroot-models/ssd300_mAP_77.43_v2.pth"
t_path = os.path.join(weights_dir, 'ssd300.pth')
if not os.path.exists(t_path):
    urllib.request.urlretrieve(url, t_path)

In [ ]:
from math import sqrt
from itertools import product

import pandas as pd
import torch
# from torch.autograd import Function
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

In [ ]:
def make_vgg():
    layers = []
    in_c = 3
    # VGG16 모듈 합성곱 층, max pooling 채널 수 정의
    cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'MC', 512, 512, 512, 'M', 512, 512, 512]
    for i in cfg:
        if i == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        elif i == 'MC':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)]
        else:
            conv2d = nn.Conv2d(in_c, i, kernel_size=3, padding=1)
            layers += [conv2d, nn.ReLU(inplace=True)]
            in_c = i
    pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
    conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
    conv7 = nn.Conv2d(1024, 1024, kernel_size=1)
    layers += [pool5, conv6, nn.ReLU(inplace=True), conv7, nn.ReLU(inplace=True)]
    return nn.ModuleList(layers)

vgg_m_test = make_vgg()
vgg_m_test

In [ ]:
def make_extras():
    layers = []
    in_c = 1024

    cfg = [256, 512, 128, 256, 128, 256, 128, 256]
    layers += [nn.Conv2d(in_c, cfg[0], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[0], cfg[1], kernel_size=(3), stride=2, padding=1)]
    layers += [nn.Conv2d(cfg[1], cfg[2], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[2], cfg[3], kernel_size=(3), stride=2, padding=1)]
    layers += [nn.Conv2d(cfg[3], cfg[4], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[4], cfg[5], kernel_size=(3))]
    layers += [nn.Conv2d(cfg[5], cfg[6], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[6], cfg[7], kernel_size=(3))]
    return nn.ModuleList(layers)
make_extras()

In [ ]:
def make_loc_conf(class_n = 21, bbox_aspect_num = [4, 6, 6, 6, 4, 4]):
        loc_layers = []
        conf_layers = []

        loc_layers += [nn.Conv2d(512, bbox_aspect_num[0]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(512, bbox_aspect_num[0]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(1024, bbox_aspect_num[1]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(1024, bbox_aspect_num[1]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(512, bbox_aspect_num[2]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(512, bbox_aspect_num[2]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(256, bbox_aspect_num[3]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(256, bbox_aspect_num[3]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(256, bbox_aspect_num[4]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(256, bbox_aspect_num[4]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(256, bbox_aspect_num[5]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(256, bbox_aspect_num[5]*class_n, kernel_size=3, padding=1)]

        return nn.ModuleList(loc_layers), nn.ModuleList(conf_layers)

make_loc_conf()

In [ ]:
class L2Norm(nn.Module):
    def __init__(self, input_c = 512, scale = 20):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(input_c))
        self.scale = scale
        self.reset_parameter()
        self.eps = 1e-10

    def reset_parameter(self):
        init.constant_(self.weight, self.scale)

    def forward(self, x):
        norm = x.pow(2).sum(dim = 1, keepdim = True).sqrt()+self.eps
        x = torch.div(x, norm)
        weights = self.weight.unsqueeze(0).unsqueeze(2).unsqueeze(3).expand_as(x)
        out = weights*x
        return out

In [ ]:
ssd_cfg = {
    'num_classes': 21,  # 배경 클래스를 포함한 총 클래스 수
    'input_size': 300,  # 화상의 입력 크기
    'bbox_aspect_num': [4, 6, 6, 6, 4, 4],  # 출력할 Box 화면비의 종류
    'feature_maps': [38, 19, 10, 5, 3, 1],  # 각 source의 화상 크기
    'steps': [8, 16, 32, 64, 100, 300],  # DBOX의 크기를 정한다
    'min_sizes': [30, 60, 111, 162, 213, 264],  # DBOX의 크기를 정한다
    'max_sizes': [60, 111, 162, 213, 264, 315],  # DBOX의 크기를 정한다
    'aspect_ratios': [[2], [2, 3], [2, 3], [2, 3], [2], [2]],
}

In [ ]:
class DBox:
    def __init__(self, cfg):
        super().__init__()
        self.img_size = cfg['input_size']
        self.feature_maps = cfg['feature_maps']
        self.num_priors = len(cfg['feature_maps'])
        self.steps = cfg['steps']
        self.min_sizes = cfg['min_sizes']
        self.max_sizes = cfg['max_sizes']
        self.aspect_ratios = cfg['aspect_ratios']

    def make_dbox_list(self):
        mean = []
        for k,f in enumerate(self.feature_maps):
            for i,j in product(range(f), repeat = 2):
                f_k = self.img_size/self.steps[k]

                cx = (j+.5)/f_k
                cy = (i+.5)/f_k

                s_k = self.min_sizes[k]/self.img_size
                mean += [cx, cy, s_k, s_k]

                s_k_prime = sqrt(s_k*(self.max_sizes[k]/self.img_size))
                mean += [cx, cy, s_k_prime, s_k_prime]

                for a in self.aspect_ratios[k]:
                    mean += [cx, cy, s_k*sqrt(a), s_k/sqrt(a)]
                    mean += [cx, cy, s_k/sqrt(a), s_k*sqrt(a)]
        output = torch.Tensor(mean).view(-1, 4)
        output.clamp_(max=1, min=0)
        return output

In [ ]:
dbox = DBox(ssd_cfg)
pd.DataFrame(dbox.make_dbox_list())

In [ ]:
class SSD(nn.Module):
    def __init__(self, phase, cfg):
        super().__init__()
        self.phase = phase
        self.num_classes = cfg['num_classes']

        self.vgg = make_vgg()
        self.extras = make_extras()
        self.L2Norm = L2Norm()
        self.loc,self.conf = make_loc_conf(cfg['num_classes'], cfg['bbox_aspect_num'])

        dbox = DBox(cfg)
        self.dbox_list = dbox.make_dbox_list()

        if phase == 'inference':
            self.detect = Detect()
ssd_test=SSD(phase='train', cfg=ssd_cfg)
ssd_test

In [ ]:
def decode(loc, dbox_list):
    boxes = torch.car((
        dbox_list[:, :2] + loc[:, :2]*.1*dbox_list[:, 2:],
        dbox_list[:, 2:] * torch.exp(loc[:, 2:] * .2)), dim=1)
    boxes[:, :2] -= boxes[:, :2] / 2
    boxes[:, 2:] += boxes[:, :2]
    return boxes

In [ ]:
def nn_suppression(boxes, scores, overlap=.45, top_k=200):
    count=0
    keep=scores.new(scores.size(0)).zero_().long

    x1 = boxes[: ,0]
    y1 = boxes[: ,1]
    x2 = boxes[: ,2]
    y2 = boxes[: ,3]
    area = torch.mul(x2-x1, y2-y1)

    tmp_x1 = boxes.new()
    tmp_y1 = boxes.new()
    tmp_x2 = boxes.new()
    tmp_y2 = boxes.new()
    tmp_w = boxes.new()
    tmp_h = boxes.new()

    v, idx = scores.sort(0)
    idx = idx[-top_k:]

    while idx.numel() > 0:
        i = idx[-1]

        keep[count] = i
        count += 1

        if idx.size(0) == 1:
            break
        idx = idx[:-1]
        torch.index_select(x1, 0, idx, out=tmp_x1)
        torch.index_select(y1, 0, idx, out=tmp_y1)
        torch.index_select(x2, 0, idx, out=tmp_x2)
        torch.index_select(y2, 0, idx, out=tmp_y2)

        tmp_x1 = torch.clamp(tmp_x1, min=x1[i])
        tmp_y1 = torch.clamp(tmp_y1, min=y1[i])
        tmp_x2 = torch.clamp(tmp_x2, min=x2[i])
        tmp_y2 = torch.clamp(tmp_y2, min=y2[i])

        tmp_w.resize_as_(tmp_x2)
        tmp_h.resize_as_(tmp_y2)

        tmp_w = tmp_x2-tmp_x1
        tmp_h = tmp_y2-tmp_y1

        tmp_w = torch.clamp(tmp_w, min=.0)
        tmp_h = torch.clamp(tmp_h, min=.0)

        inter = tmp_h * tmp_w

        rem_areas = torch.index_select(area, 0, idx)
        union = (rem_areas - inter) + area[i]
        IoU = inter / union

        idx = idx[IoU.le(overlap)]

In [ ]:
class Detect(nn.Module):
    def __init__(self, conf_thresh=.01, top_k=200, nms_thresh=.45):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.conf_thresh = conf_thresh
        self.top_k = top_k
        self.nms_thresh = nms_thresh

    def forward(self, loc_data, conf_data, dbox_list):
        num_batch = loc_data.size(0)
        num_dbox = loc_data.size(1)
        num_classes = conf_data.size(2)

        conf_data = self.softmax(conf_data)

        output = torch.zeros(num_batch, num_classes, self.top_k, 5)
        conf_preds = conf_data.transpose(2,1)

        for i in range(num_batch):
            decoded_boxes = decode(loc_data[i], dbox_list)
            conf_scores = conf_preds[i].clone()

            for cl in range(1, num_classes):
                c_mask = conf_scores[cl].gt(self.conf_thresh)
                scores = conf_scores[cl][c_mask]
                if scores.nelement() == 0:
                    continue
                l_mask = c_mask.unsqeeze(1).expand_as(decoded_boxes)
                boxes = decoded_boxes[l_mask].view(-1, 4)

                ids, count = nm_suppression(boxes, scores, self.nms_thresh, self.top_k)
                output[incl,:count] = torch.cat((scores[ids[:count]].unsqeeze(1), boxes[ids[:count]]), 1)

        return output

In [ ]:
class SSD(nn.Module):
    def __init__(self, phase, cfg):
        super().__init__()
        self.phase = phase
        self.num_classes = cfg['num_classes']

        self.vgg = make_vgg()
        self.extras = make_extras()
        self.L2Norm = L2Norm()
        self.loc,self.conf = make_loc_conf(cfg['num_classes'], cfg['bbox_aspect_num'])

        dbox = DBox(cfg)
        self.dbox_list = dbox.make_dbox_list()

        if phase == 'inference':
            self.detect = Detect()
    def forward(self, x):
        sources = []
        loc = []
        conf = []

        for k in range(23):
            x = self.vgg[k](x)
        source1 = self.L2Norm(x)
        sources.append(source1)
        for k in range(23, len(self.vgg)):
            x = self.vgg[k](x)
        sources.append(x)
        for k, v in enumerate(self.extras):
            x = F.relu(v(x), inplace=True)
            if k%2==1:
                sources.append(x)
        for (x, l, c) in zip(sources, self.loc, self.conf):
            loc.append(l(x).permute(0, 2, 3, 1).contiguous())
            conf.append(c(x).permute(0, 2, 3, 1).contiguous())
        loc = torch.cat([o.view(o.size(0), -1) for io in loc], 1)
        conf = torch.cat([o.view(o.size(0), -1) for io in conf], 1)
        loc = loc.view(loc.size(0), -1, 4)
        conf = conf.view(conf.size(0), -1, self.num_classes)

        output = (loc, conf, self.dbox_list)

        if self.phase == 'inference':
            return self.detect(output[0], output[1], output[2])
        else:
            return output

In [ ]:
class MultiBoxLoss(nn.Module):
    def __init__(self, jaccard_thresh=.5, neg_pos=3):
        super().__init__()
        self.jaccard_thresh = jaccard_thresh
        self.neg_pos = neg_pos

    def forward(self, py, ty):
        loc_data, conf_data, dbox_list = py
        n_batch = loc_data.size(0)
        n_dbox = loc_data.size(1)
        n_classes = conf_data.size(2)

        conf_t_label = torch.LongTensor(n_batch, n_dbox)
        loc_t = torch.Tensor(n_batch, n_dbox, 4)

        for idx in range(n_batch):
            truths = ty[idx][:,:,-1]
            labels = ty[idx][:,-1]
            dbox = dbox_list
            variance = [.1, .2]

            match(self.jaccard_thresh, truths, dbox, variance, labels, loc_t, conf_t_label, idx)

            # 위치 손실
            # 클래스 손실
            # 데이터 안에 있는 객체의 위치 클래스 일치성을 통합하여 loss 계산

## 기 구축된 모듈 활용 모델 학습

In [1]:
import os.path as osp
import random
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
import torch.utils.data as data

In [2]:
torch.manual_seed(1111)
np.random.seed(1111)
random.seed(1111)

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
DEVICE

device(type='mps')

In [4]:
from utils.ssd_model import *